In [ ]:
using Plots
using LinearAlgebra
using Statistics
gr()

In [ ]:
# Nomalised wavelength for ideal geometry compariso
λ = 1.0

println("Normalised wavelength λ = ", λ)

In [ ]:
# Number of antenna elements
N = 3

# Angular position of each antenna around the UCA
alpha = (0:N-1) .* (2 * π / N)

# Degrees for easier inspection
alpha_deg = rad2deg.(alpha)

println("Antenna angles = ", alpha_deg)

In [ ]:
# Adjacent antenna spacing for the first comparison
d = 0.75 * λ

# UCA radius corresponding to adjacent chord spacing d
R = d / (2 * sin(π / N))

println("Adjacent spacing d/λ = ", d / λ)
println("Array radius R/λ = ", R / λ)

In [ ]:
# Plot the three-element UCA geometry
p = scatter(
    x ./ λ, y ./ λ,
    xlabel = "x / λ",
    ylabel = "y / λ",
    label = false,
    markersize = 8,
    aspect_ratio = :equal,
    title = "Three-Element Uniform Circular Array"
)

scatter!([0], [0],
    label = false,
    markersize = 5
)

for n = 1:N
    annotate!(x[n] / λ, y[n] / λ + 0.015, text("A$n", 10))
end

display(p)

In [ ]:
# True source azimuth
theta_true_deg = 30.0
theta_true = deg2rad(theta_true_deg)

# Path difference relative to reference antenna A1
delta_r = R .* (
    cos(theta_true) .-
    cos.(theta_true .- alpha)
)

for n = 1:N
    println(
        "Antenna ", n,
        ": Δr_1", n, "/λ = ", delta_r[n] / λ
    )
end

In [ ]:
# Reference-relative phase differences for the true source direction
delta_phi = (2 * π / λ) .* delta_r

# Wrap phases to [-π, π]
delta_phi_wrapped = angle.(exp.(im .* delta_phi))

# Convert to degrees for inspection
delta_phi_deg = rad2deg.(delta_phi_wrapped)

for n = 1:N
    println(
        "Antenna ", n,
        ": Δφ_1", n, " = ", delta_phi_deg[n], " degrees"
    )
end

In [ ]:
# Simulated measured phase vector
# A1 is the reference, so only A2...AN are included
phi_meas = delta_phi_deg[2:end]

println("Measured reference-relative phase vector:")
println(phi_meas)

In [ ]:
# Candidate azimuths
theta_scan_deg = 0.0:0.1:359.9

# RMSE and direct phase-match score
J = zeros(length(theta_scan_deg))
score = zeros(length(theta_scan_deg))

# Include A1 = 0° as the phase reference
phase_meas_full = vcat(0.0, deg2rad.(phi_meas))
a_meas = exp.(im .* phase_meas_full)

for k = eachindex(theta_scan_deg)

    theta = deg2rad(theta_scan_deg[k])

    # Expected path differences relative to A1
    delta_r_exp = R .* (
        cos(theta) .-
        cos.(theta .- alpha)
    )

    # Expected reference-relative phases
    delta_phi_exp_rad = angle.(
        exp.(im .* (2 * π / λ) .* delta_r_exp)
    )

    delta_phi_exp_deg = rad2deg.(delta_phi_exp_rad[2:end])

    # -----------------------------
    # Wrapped phase-error RMSE
    # -----------------------------
    error = rad2deg.(
        angle.(exp.(im .* deg2rad.(phi_meas .- delta_phi_exp_deg)))
    )

    J[k] = sqrt(sum(error .^ 2) / (N - 1))

    # -----------------------------
    # Direct phase-match score
    # -----------------------------
    a_exp = exp.(im .* delta_phi_exp_rad)

    score[k] = abs(sum(conj.(a_exp) .* a_meas)) / N
end

# Direction estimates from both methods
theta_est_rmse = theta_scan_deg[argmin(J)]
theta_est_score = theta_scan_deg[argmax(score)]

println("True direction        = ", theta_true_deg, " degrees")
println("RMSE estimate         = ", theta_est_rmse, " degrees")
println("Phase-match estimate  = ", theta_est_score, " degrees")
println("Maximum score         = ", maximum(score))

In [ ]:
# Perfect-match locations for plotting
tolerance = 1e-6
perfect_indices = findall(J .< tolerance)
perfect_angles = theta_scan_deg[perfect_indices]

p = plot(
    deg2rad.(theta_scan_deg),
    score,
    proj = :polar,
    linewidth = 2,
    label = false,
    title = "UCA Phase Match, d = $(round(d / λ, digits=2))λ",
    titlefontsize = 11,
    rlims = (0, 1.1),
    rticks = 0:0.25:1.0,
    thetagrids = 0:45:315,
    gridalpha = 0.65,
    gridlinewidth = 0.8,
    size = (760, 600)
)
# Mark and label the exact phase-match direction(s)
match_labels = [
    "($(round(Int, theta_scan_deg[i]))°, $(round(score[i], digits=2)))"
    for i in perfect_indices
]

scatter!(
    p,
    deg2rad.(perfect_angles),
    score[perfect_indices],
    markersize = 5,
    markerstrokewidth = 1,
    markerstrokecolor = :black,
    label = false,
    series_annotations = match_labels
)

display(p)

d_name = replace(string(round(d / λ, digits=2)), "." => "p")
savefig(p, "UCA_PhaseMatch_d_$(d_name)lambda.png")

In [ ]:
tolerance = 1e-6

perfect_indices = findall(J .< tolerance)
perfect_angles = theta_scan_deg[perfect_indices]

println("Perfect-match directions = ", perfect_angles)

In [ ]:
# Fixed array radius for element-count comparison

N_ref = 3
d_ref = 0.5 * λ

R_fixed = d_ref / (2 * sin(π / N_ref))

println("Fixed radius R/λ = ", R_fixed / λ)

In [ ]:
N_values = [3, 4, 5, 6]

for N_test in N_values
    d_test = 2 * R_fixed * sin(π / N_test)
    println(
        "N = ", N_test,
        "   d/λ = ", round(d_test / λ, digits=4)
    )
end

In [ ]:
theta_true_deg = 30.0
theta_true = deg2rad(theta_true_deg)

for N_test in N_values

    alpha_test = (0:N_test-1) .* (2 * π / N_test)

    delta_r_test = R_fixed .* (
        cos(theta_true) .-
        cos.(theta_true .- alpha_test)
    )

    delta_phi_test = angle.(
        exp.(im .* (2 * π / λ) .* delta_r_test)
    )

    println(
        "N = ", N_test,
        "   phases = ",
        round.(rad2deg.(delta_phi_test[2:end]), digits=2)
    )
end

In [ ]:
theta_scan_deg = 0.0:0.1:359.9

results_by_N = Dict()

for N_test in N_values

    alpha_test = (0:N_test-1) .* (2 * π / N_test)

    # True reference-relative phases
    delta_r_true = R_fixed .* (
        cos(theta_true) .-
        cos.(theta_true .- alpha_test)
    )

    delta_phi_true = angle.(
        exp.(im .* (2 * π / λ) .* delta_r_true)
    )

    phi_meas = rad2deg.(delta_phi_true[2:end])

    J_test = zeros(length(theta_scan_deg))
    score_test = zeros(length(theta_scan_deg))

    a_meas = exp.(im .* delta_phi_true)

    for k in eachindex(theta_scan_deg)

        theta = deg2rad(theta_scan_deg[k])

        delta_r_exp = R_fixed .* (
            cos(theta) .-
            cos.(theta .- alpha_test)
        )

        delta_phi_exp = angle.(
            exp.(im .* (2 * π / λ) .* delta_r_exp)
        )

        error = rad2deg.(
            angle.(
                exp.(
                    im .* (
                        deg2rad.(phi_meas) .-
                        delta_phi_exp[2:end]
                    )
                )
            )
        )

        J_test[k] = sqrt(sum(error .^ 2) / (N_test - 1))

        a_exp = exp.(im .* delta_phi_exp)
        score_test[k] = abs(sum(conj.(a_exp) .* a_meas)) / N_test
    end

    perfect_angles = theta_scan_deg[findall(J_test .< 1e-6)]

    results_by_N[N_test] = (
        J = J_test,
        score = score_test
    )

    println(
        "N = ", N_test,
        "   estimate = ", theta_scan_deg[argmin(J_test)], "°",
        "   perfect matches = ", perfect_angles
    )
end

In [ ]:
for N_test in N_values

    J_test = results_by_N[N_test].J
    score_test = results_by_N[N_test].score

    perfect_indices = findall(J_test .< 1e-6)
    perfect_angles = theta_scan_deg[perfect_indices]

    p = plot(
        deg2rad.(theta_scan_deg),
        score_test,
        proj = :polar,
        linewidth = 2,
        label = false,
        title = "UCA Phase Match, N = $N_test",
        titlefontsize = 11,
        rlims = (0, 1.1),
        rticks = 0:0.25:1.0,
        thetagrids = 0:45:315,
        gridalpha = 0.65,
        gridlinewidth = 0.8,
        size = (760, 600)
    )

    match_labels = [
        "($(round(Int, theta_scan_deg[i]))°, $(round(score_test[i], digits=2)))"
        for i in perfect_indices
    ]

    scatter!(
        p,
        deg2rad.(perfect_angles),
        score_test[perfect_indices],
        markersize = 5,
        markerstrokewidth = 1,
        markerstrokecolor = :black,
        label = false,
        series_annotations = match_labels
    )

    display(p)

    savefig(p, "UCA_PhaseMatch_N_$(N_test).png")
end

In [ ]:
for N_test in N_values

    score_test = results_by_N[N_test].score
    L = length(score_test)

    # Find circular local maxima
    local_max_indices = Int[]

    for i in 1:L
        i_prev = mod1(i - 1, L)
        i_next = mod1(i + 1, L)

        if score_test[i] > score_test[i_prev] &&
           score_test[i] >= score_test[i_next]

            push!(local_max_indices, i)
        end
    end

    # Sort local maxima from strongest to weakest
    sorted_indices = sort(
        local_max_indices,
        by = i -> score_test[i],
        rev = true
    )

    # First maximum is the true 30° direction
    competitor_index = sorted_indices[2]

    println(
        "N = ", N_test,
        "   strongest competing angle = ",
        theta_scan_deg[competitor_index], "°",
        "   score = ",
        round(score_test[competitor_index], digits=4)
    )
end

In [ ]:
for N_test in N_values

    J_test = results_by_N[N_test].J
    L = length(J_test)

    # Find circular local minima of J
    local_min_indices = Int[]

    for i in 1:L
        i_prev = mod1(i - 1, L)
        i_next = mod1(i + 1, L)

        if J_test[i] <= J_test[i_prev] &&
           J_test[i] <= J_test[i_next] &&
           (J_test[i] < J_test[i_prev] || J_test[i] < J_test[i_next])

            push!(local_min_indices, i)
        end
    end

    # Sort minima from lowest J to highest J
    sorted_indices = sort(
        local_min_indices,
        by = i -> J_test[i]
    )

    true_index = argmin(J_test)

    # Remove the true-direction minimum
    competing_indices = filter(i -> i != true_index, sorted_indices)

    competitor_index = competing_indices[1]

    println(
        "N = ", N_test,
        "   true J = ", round(J_test[true_index], digits=4), "°",
        "   strongest competitor = ", theta_scan_deg[competitor_index], "°",
        "   competing J = ", round(J_test[competitor_index], digits=4), "°"
    )
end

## Test 3

In [ ]:
# ------------------------------------------------------------
# Array-element-count investigation
# Fixed UCA radius based on the existing 3-element, d = 0.5λ case
# ------------------------------------------------------------

N_ref = 3
d_ref = 0.5 * λ

R_fixed = d_ref / (2 * sin(π / N_ref))

N_values = 3:6

println("Fixed array radius: R/λ = ", R_fixed / λ)
println()
println(" N    adjacent spacing d/λ")
println("---------------------------")

for N in N_values
    dN = 2 * R_fixed * sin(π / N)
    println(rpad(N, 5), round(dN / λ, digits=4))
end

In [ ]:
using Random
using Statistics

# ------------------------------------------------------------
# UCA noisy-I/Q simulation and DoA estimator
# ------------------------------------------------------------

# Reference-relative phase for an N-element UCA
function uca_phases(theta_deg, N, R, λ)
    theta = deg2rad(theta_deg)
    alpha = 2π .* (0:N-1) ./ N

    return (2π * R / λ) .* (
        cos(theta) .- cos.(theta .- alpha)
    )
end


# Generate received I/Q samples
function generate_iq(theta_deg, N, R, λ, K, snr_db)
    Δphi = uca_phases(theta_deg, N, R, λ)

    # Unit-power source signal: Ps = 1
    s = ones(ComplexF64, K)

    # Noise standard deviation from Section 4.4.2
    σ = isinf(snr_db) ? 0.0 :
        sqrt(1 / (2 * 10^(snr_db / 10)))

    x = zeros(ComplexF64, K, N)

    for n in 1:N
        w = σ .* (randn(K) .+ 1im .* randn(K))

        x[:, n] =
            s .* exp(-1im * Δphi[n]) .+ w
    end

    return x
end


# Extract reference-relative phases using antenna A1
function measured_phases(x)
    K, N = size(x)

    phi_meas = zeros(N - 1)

    for n in 2:N
        C1n = mean(x[:, 1] .* conj.(x[:, n]))
        phi_meas[n - 1] = angle(C1n)
    end

    return phi_meas
end


# Estimate source direction
function estimate_doa(phi_meas, N, R, λ;
                      theta_grid = 0.0:1.0:359.0)

    J = zeros(length(theta_grid))

    for (i, theta) in enumerate(theta_grid)

        phi_exp =
            uca_phases(theta, N, R, λ)[2:end]

        # Wrapped phase error
        e = angle.(exp.(1im .* (phi_meas .- phi_exp)))

        J[i] = sqrt(mean(e .^ 2))
    end

    return theta_grid[argmin(J)]
end


# ------------------------------------------------------------
# Noise-free sanity check
# ------------------------------------------------------------

Random.seed!(4022)

theta_true = 30.0
N_test = 3
K_test = 64

x_test = generate_iq(
    theta_true,
    N_test,
    R_fixed,
    λ,
    K_test,
    Inf
)

phi_test = measured_phases(x_test)

theta_hat = estimate_doa(
    phi_test,
    N_test,
    R_fixed,
    λ
)

println("True direction      = ", theta_true, "°")
println("Estimated direction = ", theta_hat, "°")

In [ ]:
# ------------------------------------------------------------
# Effect of number of averaged I/Q samples, K
# ------------------------------------------------------------

function angular_error_deg(theta_hat, theta_true)
    return mod(theta_hat - theta_true + 180, 360) - 180
end

K_values = [16, 32, 64, 128, 256, 512]

theta_true = 30.0
N_test = 3
snr_test = -10.0

M = 300   # Monte Carlo trials

Random.seed!(4022)

rmse_K = zeros(length(K_values))

for (i, K) in enumerate(K_values)

    errors = zeros(M)

    for m in 1:M
        x = generate_iq(
            theta_true,
            N_test,
            R_fixed,
            λ,
            K,
            snr_test
        )

        phi_meas = measured_phases(x)

        theta_hat = estimate_doa(
            phi_meas,
            N_test,
            R_fixed,
            λ
        )

        errors[m] =
            angular_error_deg(theta_hat, theta_true)
    end

    rmse_K[i] = sqrt(mean(errors .^ 2))

    println(
        "K = ", rpad(K, 4),
        "  RMSE = ",
        round(rmse_K[i], digits=2),
        "°"
    )
end

In [ ]:
# ------------------------------------------------------------
# Extend K investigation
# ------------------------------------------------------------

K_values_ext = [512, 1024, 2048, 4096]

theta_true = 30.0
N_test = 3
snr_test = -10.0
M = 300

Random.seed!(4022)

for K in K_values_ext

    errors = zeros(M)

    for m in 1:M
        x = generate_iq(
            theta_true,
            N_test,
            R_fixed,
            λ,
            K,
            snr_test
        )

        phi_meas = measured_phases(x)

        theta_hat = estimate_doa(
            phi_meas,
            N_test,
            R_fixed,
            λ
        )

        errors[m] =
            angular_error_deg(theta_hat, theta_true)
    end

    rmse = sqrt(mean(errors .^ 2))
    failure_rate = 100 * mean(abs.(errors) .> 30)

    println(
        "K = ", rpad(K, 5),
        " RMSE = ", rpad(round(rmse, digits=2), 7), "°",
        "  |error| > 30° = ",
        round(failure_rate, digits=1), "%"
    )
end

In [ ]:
# ------------------------------------------------------------
# Element-count performance versus SNR
# ------------------------------------------------------------

N_values = 3:6
SNR_values = [-20.0, -15.0, -10.0, -5.0, 0.0]

theta_true = 30.0
K = 2048
M = 1000

Random.seed!(4022)

rmse_results = zeros(length(SNR_values), length(N_values))
failure_results = zeros(length(SNR_values), length(N_values))

for (i_snr, snr_db) in enumerate(SNR_values)

    for (i_N, N) in enumerate(N_values)

        errors = zeros(M)

        for m in 1:M
            x = generate_iq(
                theta_true,
                N,
                R_fixed,
                λ,
                K,
                snr_db
            )

            phi_meas = measured_phases(x)

            theta_hat = estimate_doa(
                phi_meas,
                N,
                R_fixed,
                λ
            )

            errors[m] =
                angular_error_deg(theta_hat, theta_true)
        end

        rmse_results[i_snr, i_N] =
            sqrt(mean(errors .^ 2))

        failure_results[i_snr, i_N] =
            100 * mean(abs.(errors) .> 30)

        println(
            "SNR = ", rpad(snr_db, 6),
            " dB   N = ", N,
            "   RMSE = ",
            round(rmse_results[i_snr, i_N], digits=2),
            "°   |error| > 30° = ",
            round(failure_results[i_snr, i_N], digits=1),
            "%"
        )
    end

    println()
end

In [ ]:
# ------------------------------------------------------------
# Check element-count performance across source azimuth
# ------------------------------------------------------------

theta_values = 0.0:30.0:330.0
N_values = 3:6

snr_test = -15.0
K = 2048
M = 300

Random.seed!(4022)

rmse_angle = zeros(length(theta_values), length(N_values))

for (i_theta, theta_true) in enumerate(theta_values)

    for (i_N, N) in enumerate(N_values)

        errors = zeros(M)

        for m in 1:M
            x = generate_iq(
                theta_true,
                N,
                R_fixed,
                λ,
                K,
                snr_test
            )

            phi_meas = measured_phases(x)

            theta_hat = estimate_doa(
                phi_meas,
                N,
                R_fixed,
                λ
            )

            errors[m] =
                angular_error_deg(theta_hat, theta_true)
        end

        rmse_angle[i_theta, i_N] =
            sqrt(mean(errors .^ 2))
    end

    println(
        "θ = ", rpad(theta_true, 5), "°   ",
        "N3 = ", round(rmse_angle[i_theta,1], digits=1), "°   ",
        "N4 = ", round(rmse_angle[i_theta,2], digits=1), "°   ",
        "N5 = ", round(rmse_angle[i_theta,3], digits=1), "°   ",
        "N6 = ", round(rmse_angle[i_theta,4], digits=1), "°"
    )
end

In [ ]:
# ------------------------------------------------------------
# Summary of azimuth test
# ------------------------------------------------------------

println("Mean of per-angle RMSE values at -15 dB")
println("-----------------------------------------")

for (i_N, N) in enumerate(N_values)
    println(
        "N = ", N,
        "   mean RMSE = ",
        round(mean(rmse_angle[:, i_N]), digits=2),
        "°"
    )
end

In [ ]:
# ------------------------------------------------------------
# Final element-count comparison
# Pooled across source azimuth and Monte Carlo trials
# ------------------------------------------------------------

N_values = 3:6
SNR_values = [-20.0, -15.0, -10.0, -5.0, 0.0]
theta_values = 0.0:30.0:330.0

K = 2048
M = 300

Random.seed!(4022)

rmse_overall = zeros(length(SNR_values), length(N_values))
failure_overall = zeros(length(SNR_values), length(N_values))

for (i_snr, snr_db) in enumerate(SNR_values)

    for (i_N, N) in enumerate(N_values)

        # Store errors from every azimuth and trial
        errors = Float64[]

        for theta_true in theta_values

            for m in 1:M

                x = generate_iq(
                    theta_true,
                    N,
                    R_fixed,
                    λ,
                    K,
                    snr_db
                )

                phi_meas = measured_phases(x)

                theta_hat = estimate_doa(
                    phi_meas,
                    N,
                    R_fixed,
                    λ
                )

                push!(
                    errors,
                    angular_error_deg(theta_hat, theta_true)
                )
            end
        end

        rmse_overall[i_snr, i_N] =
            sqrt(mean(errors .^ 2))

        failure_overall[i_snr, i_N] =
            100 * mean(abs.(errors) .> 30)

        println(
            "SNR = ", rpad(snr_db, 6), " dB",
            "   N = ", N,
            "   RMSE = ",
            round(rmse_overall[i_snr, i_N], digits=2),
            "°   |error| > 30° = ",
            round(failure_overall[i_snr, i_N], digits=1),
            "%"
        )
    end

    println()
end

In [ ]:
# ------------------------------------------------------------
# Final uniform-azimuth Monte Carlo element-count study
# ------------------------------------------------------------

N_final = 3:10
SNR_final = [-15.0, -10.0]

K = 2048
M = 3000

Random.seed!(4022)

# Same uniformly distributed source directions for all cases
theta_trials = 360 .* rand(M)

rmse_final = zeros(length(SNR_final), length(N_final))
failure_final = zeros(length(SNR_final), length(N_final))

# Keep the individual errors so that confidence intervals
# can be calculated afterwards
error_store = Dict{Tuple{Float64, Int}, Vector{Float64}}()

for (i_snr, snr_db) in enumerate(SNR_final)

    println("\nSNR = $(snr_db) dB")
    println("---------------------------------------------")

    for (i_N, N) in enumerate(N_final)

        errors = zeros(M)

        for m in 1:M

            theta_true = theta_trials[m]

            x = generate_iq(
                theta_true,
                N,
                R_fixed,
                λ,
                K,
                snr_db
            )

            phi_meas = measured_phases(x)

            theta_hat = estimate_doa(
                phi_meas,
                N,
                R_fixed,
                λ
            )

            errors[m] =
                angular_error_deg(theta_hat, theta_true)
        end

        error_store[(snr_db, N)] = copy(errors)

        rmse_final[i_snr, i_N] =
            sqrt(mean(errors .^ 2))

        failure_final[i_snr, i_N] =
            100 * mean(abs.(errors) .> 30)

        println(
            "N = ", rpad(N, 3),
            " RMSE = ",
            rpad(round(rmse_final[i_snr, i_N], digits=2), 7),
            "°   |error| > 30° = ",
            round(failure_final[i_snr, i_N], digits=2),
            "%"
        )
    end
end

In [ ]:
# ------------------------------------------------------------
# Bootstrap 95% confidence intervals for RMSE
# Uses the stored Monte Carlo errors — no new DoA simulations
# ------------------------------------------------------------

Random.seed!(4022)

B = 1000   # bootstrap repetitions

rmse_lower = zeros(length(SNR_final), length(N_final))
rmse_upper = zeros(length(SNR_final), length(N_final))

for (i_snr, snr_db) in enumerate(SNR_final)

    println("\nSNR = $(snr_db) dB")
    println("---------------------------------------------")

    for (i_N, N) in enumerate(N_final)

        errors = error_store[(snr_db, N)]
        M_err = length(errors)

        boot_rmse = zeros(B)

        for b in 1:B
            idx = rand(1:M_err, M_err)

            boot_rmse[b] =
                sqrt(mean(errors[idx] .^ 2))
        end

        rmse_lower[i_snr, i_N] = quantile(boot_rmse, 0.025)
        rmse_upper[i_snr, i_N] = quantile(boot_rmse, 0.975)

        println(
            "N = ", N,
            "   RMSE = ",
            round(rmse_final[i_snr, i_N], digits=2),
            "°   95% CI = [",
            round(rmse_lower[i_snr, i_N], digits=2),
            "°, ",
            round(rmse_upper[i_snr, i_N], digits=2),
            "°]"
        )
    end
end

In [ ]:
using Plots

# Error-bar distances
err_low_15  = rmse_final[1, :] .- rmse_lower[1, :]
err_high_15 = rmse_upper[1, :] .- rmse_final[1, :]

err_low_10  = rmse_final[2, :] .- rmse_lower[2, :]
err_high_10 = rmse_upper[2, :] .- rmse_final[2, :]

# -----------------------------
# Plot for SNR = -15 dB
# -----------------------------
p_15 = plot(
    N_final,
    rmse_final[1, :],
    yerror = (err_low_15, err_high_15),
    marker = :circle,
    linewidth = 2,
    xlabel = "Number of elements, N",
    ylabel = "DoA RMSE (°)",
    title = "SNR = -15 dB",
    label = "",
    xticks = collect(N_final),
    grid = true,
    size = (550, 400),
    left_margin = 7Plots.mm,
    bottom_margin = 7Plots.mm
)

display(p_15)
savefig(p_15, "element_count_rmse_snr_minus15dB.png")


# -----------------------------
# Plot for SNR = -10 dB
# -----------------------------
p_10 = plot(
    N_final,
    rmse_final[2, :],
    yerror = (err_low_10, err_high_10),
    marker = :circle,
    linewidth = 2,
    xlabel = "Number of elements, N",
    ylabel = "DoA RMSE (°)",
    title = "SNR = -10 dB",
    label = "",
    xticks = collect(N_final),
    grid = true,
    size = (550, 400),
    left_margin = 7Plots.mm,
    bottom_margin = 7Plots.mm
)

display(p_10)
savefig(p_10, "element_count_rmse_snr_minus10dB.png")